# From a month to its working days

**Question.** How many working days does a month have in a region, how does that change with the region, and what does a person's leave leave of it?

Many rules count days: accrual, availability, capacity. The core library holds the calendar once. [`period_range`](../src/halvaro/core/periods.py) lists months, [`month_end`](../src/halvaro/core/periods.py) gives the last day of a month, [`public_holidays`](../src/halvaro/core/calendar.py) the public holidays of a region and [`working_days`](../src/halvaro/core/calendar.py) the working days between two dates; [`net_working_days`](../src/halvaro/core/calendar.py) takes a person's leave off. The notebook builds its own small frames, so it needs no data.

In [ ]:
import datetime as dt

import numpy as np
import pandas as pd

from halvaro.core import month_end, period_range, public_holidays, working_days
from halvaro.core.calendar import net_working_days

months = period_range("2026-03", "2026-05")
months

Step 1. The working days of each of the three months in Denmark and in Hamburg. Easter and the other holidays fall differently in the two regions.

In [ ]:
rows = []
for month in months:
    first, last = dt.date(int(month[:4]), int(month[5:]), 1), month_end(month)
    rows.append({"month": month, "DK": working_days(first, last, "DK"), "DE-HH": working_days(first, last, "DE-HH")})
calendar = pd.DataFrame(rows)
calendar

Step 2. One person on leave for a week of April. The leave counts only on working days, so a holiday inside the week is not counted twice.

In [ ]:
persons = pd.DataFrame({"emp_key": ["EMP-1"], "region": ["DK"]})
leave = pd.DataFrame({"emp_key": ["EMP-1"], "leave_type": ["holiday"], "start": [dt.date(2026, 3, 30)], "end": [dt.date(2026, 4, 3)]})
net = net_working_days(persons, leave, dt.date(2026, 3, 1), dt.date(2026, 4, 30))
net

Check. The working days equal a plain numpy count of weekdays less the public holidays of the region. The net days are the working days less the leave days.

In [ ]:
def by_numpy(first, last, region):
    days = np.busday_count(first, last + dt.timedelta(days=1), holidays=sorted(public_holidays(first.year, region)))
    return int(days)

for month, dk in zip(calendar["month"], calendar["DK"]):
    assert dk == by_numpy(dt.date(int(month[:4]), int(month[5:]), 1), month_end(month), "DK")
row = net.iloc[0]
holidays = sorted(public_holidays(2026, "DK"))
assert row["working_days"] == int(np.busday_count(dt.date(2026, 3, 1), dt.date(2026, 5, 1), holidays=holidays))
assert row["leave_days"] == int(np.busday_count(dt.date(2026, 3, 30), dt.date(2026, 4, 4), holidays=holidays))
assert row["net_days"] == row["working_days"] - row["leave_days"]